# DMM-WcycleGAN Full Pipeline Entry

This notebook now uses in-notebook parameter definitions instead of command-line strings for the local Python pipeline. The overall flow is:

1. Download the public NLB `MC_Maze_Small` dataset.
2. Define preprocessing parameters in the notebook and call `dataset_loader.py` as a module.
3. Define training parameters in the notebook and call `trainAndEval.py` as a module.
4. Read back metrics, histories, and artifacts.

Only the DANDI download step still uses an external command, because it depends on the official CLI.

In [1]:
from __future__ import annotations

import importlib.util
import json
import os
import sys
from pathlib import Path
import subprocess


def locate_work_dir() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "dataset_loader.py").exists() and (cwd / "trainAndEval.py").exists():
        return cwd

    candidate = cwd / "GAN" / "DMM-WcycleGAN"
    if (candidate / "dataset_loader.py").exists() and (candidate / "trainAndEval.py").exists():
        return candidate

    raise FileNotFoundError("Could not locate GAN/DMM-WcycleGAN from the current working directory.")


def load_module(module_name: str, module_path: Path):
    spec = importlib.util.spec_from_file_location(module_name, module_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module


WORK_DIR = locate_work_dir()
REPO_ROOT = WORK_DIR.parent.parent
DANDI_BIN = REPO_ROOT / ".venv" / "bin" / "dandi"

DATASET_DIR = WORK_DIR / "datasets" / "mc_maze_small"
RAW_DANDI_DIR = DATASET_DIR / "000140"
PIPELINE_DIR = WORK_DIR / "processed" / "mc_maze_small"
OFFICIAL_DIR = PIPELINE_DIR / "official_h5"
GAN_READY_DIR = PIPELINE_DIR / "gan_ready"
TRAIN_OUTPUT_DIR = WORK_DIR / "results" / "mc_maze_small_demo"

dataset_loader = load_module("dataset_loader_runtime", WORK_DIR / "dataset_loader.py")
train_eval = load_module("train_eval_runtime", WORK_DIR / "trainAndEval.py")
dmm_module = load_module("dmm_runtime", WORK_DIR / "DMM-WcycleGAN.py")

print(f"WORK_DIR: {WORK_DIR}")
print(f"REPO_ROOT: {REPO_ROOT}")
print(f"DATASET_DIR: {DATASET_DIR}")
print(f"GAN_READY_DIR: {GAN_READY_DIR}")
print(f"TRAIN_OUTPUT_DIR: {TRAIN_OUTPUT_DIR}")
print(f"DANDI exists: {DANDI_BIN.exists()}")

WORK_DIR: /Users/fangablt/Applications/EngineeringWorks/testProject/test_newPyEnv/GAN/DMM-WcycleGAN
REPO_ROOT: /Users/fangablt/Applications/EngineeringWorks/testProject/test_newPyEnv
DATASET_DIR: /Users/fangablt/Applications/EngineeringWorks/testProject/test_newPyEnv/GAN/DMM-WcycleGAN/datasets/mc_maze_small
GAN_READY_DIR: /Users/fangablt/Applications/EngineeringWorks/testProject/test_newPyEnv/GAN/DMM-WcycleGAN/processed/mc_maze_small/gan_ready
TRAIN_OUTPUT_DIR: /Users/fangablt/Applications/EngineeringWorks/testProject/test_newPyEnv/GAN/DMM-WcycleGAN/results/mc_maze_small_demo
DANDI exists: True


In [2]:
def local_cli_env() -> dict[str, str]:
    env = os.environ.copy()
    env["HOME"] = str(REPO_ROOT)
    env["XDG_CACHE_HOME"] = str(REPO_ROOT / ".cache")
    env["XDG_STATE_HOME"] = str(REPO_ROOT / ".state")
    env["XDG_CONFIG_HOME"] = str(REPO_ROOT / ".config")
    env["PYTHONUNBUFFERED"] = "1"
    return env


DATASET_DIR.mkdir(parents=True, exist_ok=True)

if not RAW_DANDI_DIR.exists():
    subprocess.run(
        [
            str(DANDI_BIN),
            "download",
            "-o",
            str(DATASET_DIR),
            "DANDI:000140/draft",
        ],
        cwd=REPO_ROOT,
        env=local_cli_env(),
        check=True,
    )
else:
    print("Dataset already present, skipping download.")

sorted(path.relative_to(DATASET_DIR) for path in DATASET_DIR.rglob("*.nwb"))

Dataset already present, skipping download.


[PosixPath('000140/sub-Jenkins/sub-Jenkins_ses-small_desc-test_ecephys.nwb'),
 PosixPath('000140/sub-Jenkins/sub-Jenkins_ses-small_desc-train_behavior+ecephys.nwb')]

In [3]:
dataset_loader.configure_logging(True)
PIPELINE_DIR.mkdir(parents=True, exist_ok=True)
OFFICIAL_DIR.mkdir(parents=True, exist_ok=True)
GAN_READY_DIR.mkdir(parents=True, exist_ok=True)

preprocess_params = {
    "dataset_name": "mc_maze_small",
    "nwb_path": RAW_DANDI_DIR,
    "output_dir": OFFICIAL_DIR,
    "bin_width_ms": 5,
    "include_behavior": True,
    "include_forward_pred": True,
    "eval_splits": ("val",),
    "build_full_h5": True,
}
preprocess_config = dataset_loader.PreprocessConfig(**preprocess_params)
generated_files = dataset_loader.preprocess_nlb_from_nwb(preprocess_config)

dataset_name = dataset_loader.normalize_dataset_name(preprocess_config.dataset_name)
full_h5 = OFFICIAL_DIR / f"{dataset_name}_full.h5"
export_input = full_h5 if full_h5.exists() else generated_files[-1]

export_params = {
    "input_h5": export_input,
    "output_dir": GAN_READY_DIR,
    "flatten_time": True,
}
export_config = dataset_loader.ExportConfig(**export_params)
summary_path = dataset_loader.export_h5_to_gan_ready(export_config)

preprocess_summary = json.loads(summary_path.read_text(encoding="utf-8"))
preprocess_summary

INFO: Multiple NWB files detected under `/Users/fangablt/Applications/EngineeringWorks/testProject/test_newPyEnv/GAN/DMM-WcycleGAN/datasets/mc_maze_small/000140`; defaulting to training file `sub-Jenkins_ses-small_desc-train_behavior+ecephys.nwb`.
INFO: Using NLB dataset `mc_maze_small` from `/Users/fangablt/Applications/EngineeringWorks/testProject/test_newPyEnv/GAN/DMM-WcycleGAN/datasets/mc_maze_small/000140/sub-Jenkins/sub-Jenkins_ses-small_desc-train_behavior+ecephys.nwb`
DEBUG: Registering codec 'zlib'
DEBUG: Registering codec 'gzip'
DEBUG: Registering codec 'bz2'
DEBUG: Registering codec 'lzma'
DEBUG: Registering codec 'blosc'
DEBUG: Registering codec 'zstd'
DEBUG: Registering codec 'lz4'
DEBUG: Registering codec 'astype'
DEBUG: Registering codec 'delta'
DEBUG: Registering codec 'quantize'
DEBUG: Registering codec 'fixedscaleoffset'
DEBUG: Registering codec 'packbits'
DEBUG: Registering codec 'categorize'
DEBUG: Registering codec 'pickle'
DEBUG: Registering codec 'base64'
DEBUG: 

{'input_h5': '/Users/fangablt/Applications/EngineeringWorks/testProject/test_newPyEnv/GAN/DMM-WcycleGAN/processed/mc_maze_small/official_h5/mc_maze_small_full.h5',
 'flatten_time': True,
 'splits': {'train': {'file': '/Users/fangablt/Applications/EngineeringWorks/testProject/test_newPyEnv/GAN/DMM-WcycleGAN/processed/mc_maze_small/gan_ready/train.npz',
   'arrays': {'encoder_input': [75, 140, 107],
    'heldout_target': [75, 140, 35],
    'full_target': [75, 140, 142],
    'forward_target': [75, 40, 142],
    'heldout_mask': [75, 140, 35],
    'forward_mask': [75, 40, 142],
    'extra__behavior': [75, 140, 2],
    'encoder_input_flat': [75, 14980],
    'full_target_flat': [75, 19880]}},
  'eval': {'file': '/Users/fangablt/Applications/EngineeringWorks/testProject/test_newPyEnv/GAN/DMM-WcycleGAN/processed/mc_maze_small/gan_ready/eval.npz',
   'arrays': {'encoder_input': [25, 140, 107],
    'heldout_target': [25, 140, 35],
    'full_target': [25, 140, 142],
    'forward_target': [25, 0, 1

## Training, Validation, and Evaluation

This section also uses in-notebook parameter definitions. You can edit the `train_params` dictionary directly and rerun only this cell.

In [4]:
TRAIN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_params = {
    "gan_ready_dir": GAN_READY_DIR,
    "output_dir": TRAIN_OUTPUT_DIR,
    "num_classes": 3,
    "feature_channels": 80,
    "feature_bins": 8,
    "label_topk_bins": 10,
    "train_ratio": 0.6,
    "val_ratio": 0.2,
    "seed": 42,
    "meta_tasks": 2,
    "meta_epochs": 2,
    "meta_inner_steps": 1,
    "trans_epochs": 6,
    "cnn_epochs": 12,
    "batch_size": 8,
    "generator_channels": 32,
    "critic_channels": 32,
    "classifier_channels": 32,
    "device": "auto",
}

train_config = train_eval.TrainEvalConfig(**train_params)
train_summary = train_eval.run_training(train_config, verbose=True)
train_summary

INFO: Online fine-tuning took 1.9779 seconds across 6 epochs
INFO: Train accuracy: 0.5111
INFO: Val accuracy: 0.6429
INFO: Test accuracy: 0.4375
INFO: Artifacts saved to /Users/fangablt/Applications/EngineeringWorks/testProject/test_newPyEnv/GAN/DMM-WcycleGAN/results/mc_maze_small_demo


{'config': {'gan_ready_dir': '/Users/fangablt/Applications/EngineeringWorks/testProject/test_newPyEnv/GAN/DMM-WcycleGAN/processed/mc_maze_small/gan_ready',
  'output_dir': '/Users/fangablt/Applications/EngineeringWorks/testProject/test_newPyEnv/GAN/DMM-WcycleGAN/results/mc_maze_small_demo',
  'num_classes': 3,
  'feature_channels': 80,
  'feature_bins': 8,
  'label_topk_bins': 10,
  'train_ratio': 0.6,
  'val_ratio': 0.2,
  'seed': 42,
  'meta_tasks': 2,
  'meta_epochs': 2,
  'meta_inner_steps': 1,
  'trans_epochs': 6,
  'cnn_epochs': 12,
  'batch_size': 8,
  'generator_channels': 32,
  'critic_channels': 32,
  'classifier_channels': 32,
  'device': 'auto',
  'device_resolved': 'cpu'},
 'label_metadata': {'label_mode': 'direction_sector',
  'num_classes': 3,
  'topk_bins': 10,
  'class_counts': {'0': 22, '1': 31, '2': 22}},
 'adapter': {'selected_channels': [7,
   59,
   88,
   39,
   52,
   80,
   40,
   61,
   21,
   93,
   57,
   86,
   34,
   35,
   8,
   82,
   69,
   44,
   41,
 

In [5]:
metrics = train_summary["metrics"]
print("Train metrics:")
print(json.dumps(metrics["train"], ensure_ascii=False, indent=2))

print("\nValidation metrics:")
print(json.dumps(metrics["val"], ensure_ascii=False, indent=2))

print("\nTest metrics:")
print(json.dumps(metrics["test"], ensure_ascii=False, indent=2))

print("\nCheckpoint selection:")
print(json.dumps(train_summary["checkpoint_selection"], ensure_ascii=False, indent=2))

Train metrics:
{
  "accuracy": 0.5111111111111111,
  "macro_f1": 0.3936507936507936,
  "confusion_matrix": [
    [
      9,
      4,
      0
    ],
    [
      5,
      14,
      0
    ],
    [
      8,
      5,
      0
    ]
  ]
}

Validation metrics:
{
  "accuracy": 0.6428571428571429,
  "macro_f1": 0.5,
  "confusion_matrix": [
    [
      4,
      0,
      0
    ],
    [
      1,
      5,
      0
    ],
    [
      3,
      1,
      0
    ]
  ]
}

Test metrics:
{
  "accuracy": 0.4375,
  "macro_f1": 0.3479853479853479,
  "confusion_matrix": [
    [
      3,
      2,
      0
    ],
    [
      2,
      4,
      0
    ],
    [
      4,
      1,
      0
    ]
  ]
}

Checkpoint selection:
{
  "best_epoch": 10,
  "best_val_accuracy": 0.6428571428571429
}


In [6]:
artifact_listing = sorted(path.relative_to(TRAIN_OUTPUT_DIR) for path in TRAIN_OUTPUT_DIR.rglob("*") if path.is_file())
artifact_listing

[PosixPath('.mplconfig/fontlist-v390.json'),
 PosixPath('checkpoint.pt'),
 PosixPath('eval_predictions.npz'),
 PosixPath('histories.json'),
 PosixPath('plots/classifier_history.png'),
 PosixPath('plots/confusion_matrices.png'),
 PosixPath('plots/eval_prediction_distribution.png'),
 PosixPath('plots/fine_tune_history.png'),
 PosixPath('plots/label_distribution.png'),
 PosixPath('split_indices.npz'),
 PosixPath('summary.json'),
 PosixPath('test_predictions.npz'),
 PosixPath('timing.json')]

In [7]:
histories = json.loads((TRAIN_OUTPUT_DIR / "histories.json").read_text(encoding="utf-8"))
print("Meta history tail:")
print(histories["meta"][-2:])

print("\nFine-tune history tail:")
print(histories["fine_tune"][-2:])

print("\nClassifier history tail:")
print(histories["classifier"][-3:])

Meta history tail:
[{'epoch': 1.0, 'meta_query_loss': 219.96436309814453, 'meta_critic_loss': 5.824811697006226}, {'epoch': 2.0, 'meta_query_loss': 258.29935455322266, 'meta_critic_loss': 5.476345777511597}]

Fine-tune history tail:
[{'epoch': 5.0, 'generator_loss': 12.510339736938477, 'critic_loss': 4.073639869689941}, {'epoch': 6.0, 'generator_loss': 6.4530768394470215, 'critic_loss': 3.1693763732910156}]

Classifier history tail:
[{'epoch': 10.0, 'train_loss': 1.0720250606536865, 'val_accuracy': 0.6428571428571429, 'val_macro_f1': 0.5}, {'epoch': 11.0, 'train_loss': 1.0450423657894135, 'val_accuracy': 0.42857142857142855, 'val_macro_f1': 0.19999999999999998}, {'epoch': 12.0, 'train_loss': 1.1457294523715973, 'val_accuracy': 0.42857142857142855, 'val_macro_f1': 0.19999999999999998}]


## Optional Direct Model Smoke Test

The smoke test is also switched to local parameter definitions instead of a shell command.

In [8]:
smoke_params = {
    "generator_channels": 32,
    "critic_channels": 32,
    "classifier_channels": 32,
}
smoke_config = dmm_module.DMMWcycleGANConfig(**smoke_params)
dmm_module.run_smoke_test(smoke_config)

device: cpu
generator_params: 149826
critic_params: 185602
classifier_params: 101571
generator_loss: 42.679771423339844
critic_loss: 5.906894207000732
classifier_loss: 1.0653222799301147
generator_stats: {'total': 42.679771423339844, 'adv': 0.0049219802021980286, 'cycle': 3.0327587127685547, 'identity': 2.469451904296875}
critic_stats: {'total': 5.906894207000732, 'critic_target': -0.00041034072637557983, 'critic_source': -0.028892159461975098, 'gp_target': 0.9900157451629639, 'gp_source': 0.9887164831161499}
classifier_stats: {'total': 1.0653222799301147, 'ce': 1.0650125741958618, 'reg': 3.0972745418548584}
augmented_feature_shape: (16, 1, 80, 8)
augmented_label_shape: (16,)
